In [ ]:
# 텍스트 전처리
# Tokenization(토큰화): 주어진 텍스트를 각 단어 기준으로 분리하는 것을 의미(기준은 공백)

import re  # 정규표현식 기능을 쓰기 위해 re 모듈을 불러옴

word = "123hello993 $!@eli$@ce^"  # 처리할 원본 문자열
regex = re.compile('[^a-z A-Z]')  # 소문자, 공백, 대문자가 아닌 모든 문자(숫자·특수기호)에 매칭되는 패턴을 컴파일

print(regex.sub(',', word))  # 매칭된 문자를 각각 ','로 치환(''로 하면 단어가 붙어버리는 등의 위험이 있어서 ,같은 걸로 치환)
# 결과: ",,,hello,,, ,,,eli,,ce,"
# 주의: sub은 매칭된 문자 하나하나를 개별로 바꾸는 거라, 연속된 특수문자/숫자 자리마다 콤마가 여러 개 찍힘
# (지난번 슬라이드처럼 ''로 치환하면 그냥 삭제되지만, ','로 치환하면 삭제된 자리가 콤마로 남음)

In [ ]:
import nltk  # nltk 라이브러리 불러오기
from nltk.corpus import stopwords  # nltk 내장 불용어 코퍼스에서 stopwords 모듈 불러오기

nltk.download('stopwords')  # 불용어 데이터를 처음 한 번은 다운로드해야 함 (이미 받아놨으면 생략 가능)

sentence = ["the", "green", "egg", "and", "ham", "a", "an"]  # 토큰화된 예문 (단어 리스트)
new_stopwords = ["none", "는", "가"]  # 기본 불용어 목록에 추가로 넣을 커스텀 단어들

stop_words = stopwords.words('english')  # 영어 기본 불용어 리스트 반환 ('english'는 반드시 소문자여야 함, 대문자 'English'는 에러남)
stop_words += new_stopwords  # 기본 불용어 리스트에 커스텀 불용어를 합침

new_sentence = [word for word in sentence if word not in stop_words]  # 불용어에 포함되지 않은 단어만 남김

print(new_sentence)  # ['green', 'egg', 'ham']

In [ ]:
# 단어 임베딩: 유사한 단어의 임베딩 벡터는 인접한 공간에 위치
# word2vec: 단어 간 문맥을 사용하여, 주어진 문맥에서 어떤 단어가 발생하는지 예측하는 문제로 단어 벡터를 학습
from gensim.models import Word2Vec  # gensim에서 Word2Vec 클래스를 불러옴
from gensim.models import FastText  # gensim에서 FastText 클래스를 불러옴 (word2vec 자리에 이것만 바꿔치기하면 됨)

sentences = [  # word2vec 학습에 쓸 데이터: 문장을 미리 단어 단위로 쪼개둔 리스트의 리스트
  ["i", "am", "happy", "today"],
  ["i", "feel", "sad", "and", "lonely"],
  ["this", "movie", "is", "good"],
  ["this", "movie", "is", "bad"],
]

model = Word2Vec(window=2, vector_size=300, min_count=1)  # min_count=1: 데이터가 작아서 1번만 나와도 단어로 인정하도록 설정
model.build_vocab(sentences)  # 데이터 안의 모든 단어를 훑어서 단어 사전(vocabulary)을 만듦
model.train(sentences, total_examples=model.corpus_count, epochs=10)  # 만든 사전을 바탕으로 실제 벡터를 학습 (epochs = 전체 데이터를 몇 번 반복 학습할지)

similar_words = model.wv.most_similar("happy", topn=10)  # "happy"와 벡터가 가장 가까운 단어 상위 10개를 반환
print(similar_words)

similarity_score = model.wv.similarity("good", "bad")  # 두 단어의 벡터 간 코사인 유사도(-1~1)를 계산
print(similarity_score)

vector = model.wv["happy"]  # "happy" 단어의 임베딩 벡터(숫자 배열) 자체를 가져옴
print(vector)

# fastText는 미등록 단어도 학습할 수 있다.
model = FastText(window=2, vector_size=300, min_count=1)  # window/vector_size/min_count는 word2vec과 완전히 동일한 개념
model.build_vocab(sentences)  # 사전 구축, word2vec과 동일한 절차
model.train(sentences, total_examples=model.corpus_count, epochs=10)  # 학습, word2vec과 동일한 절차

similar_words = model.wv.most_similar("happy", topn=10)  # "happy"와 유사한 단어 상위 10개
print(similar_words)

similarity_score = model.wv.similarity("good", "bad")  # 두 단어 벡터 간 코사인 유사도
print(similarity_score)

oov_vector = model.wv["happiness"]  # 학습 데이터에 단 한 번도 없던 단어인데도 벡터가 나옴
print(oov_vector)
# word2vec.wv["happiness"]였다면 여기서 KeyError가 남 (미등록단어라서)
# fastText는 단어를 문자 단위로 쪼개서 벡터를 조합하기 때문에 처음 보는 단어도 벡터를 만들어냄

In [ ]:
import math
from collections import Counter

# 학습 데이터: 문장과 그 문장의 감정 라벨
doc = ["i am very happy", "this product is really great"]
emotion = ["happy", "excited"]

# 감정별로 등장한 단어를 전부 모음 (감정별 단어 빈도수를 세기 위한 준비)
words_by_emotion = {}
for sentence, label in zip(doc, emotion):
  words = sentence.split()
  words_by_emotion.setdefault(label, []).extend(words)

# 감정별 단어 빈도수 (Counter) -> 가능도 계산의 재료
word_counts = {label: Counter(words) for label, words in words_by_emotion.items()}

# 전체 학습 데이터에 등장한 고유 단어 집합 (스무딩 분모 보정에 필요)
vocab = set(word for words in words_by_emotion.values() for word in words)

def likelihood(word, label):
  # P̂(단어|감정) = (감정 내 단어 빈도수 + 1) / (감정 내 전체 단어 빈도수 + 전체 단어 종류 수)
  # +1과 +len(vocab): 라플라스 스무딩. 학습 데이터에 없는 단어라도 확률이 0이 되지 않게 보정
  # (PDF 슬라이드 공식은 분모에 +1만 되어 있는데, 이건 확률 합이 1이 안 되는 단순화된 표기입니다.
  #  실제 sklearn의 MultinomialNB는 분모에 전체 단어 종류 수(vocab 크기)를 더하는 방식으로 동작합니다.)
  word_count = word_counts[label][word]
  total_count = sum(word_counts[label].values())
  return (word_count + 1) / (total_count + len(vocab))

def prior(label):
  # P(감정) = 그 감정을 표현하는 문서 수 / 전체 문서 수
  return emotion.count(label) / len(emotion)

def predict(sentence):
  words = sentence.split()
  scores = {}
  for label in set(emotion):
    # 로그를 쓰는 이유: 단어별 확률(0~1)을 계속 곱하면 문장이 길어질수록 값이
    # 컴퓨터가 표현 가능한 소수점 범위보다 작아짐(underflow). log(a*b)=log(a)+log(b)
    # 성질을 이용해 곱셈을 덧셈으로 바꿔서 계산
    log_prob = math.log(prior(label))
    for word in words:
      log_prob += math.log(likelihood(word, label))
    scores[label] = log_prob
  # 감정별 로그확률 중 가장 큰 값을 가진 감정을 정답으로 예측 (베이즈 정리의 최종 목적)
  return max(scores, key=scores.get)

print(predict("i am really great"))

# 한국어 자연어 처리 및 문장 유사도 정리

수업 자료 "한국어 자연어 처리 및 문장 유사도.pdf" 필터링 요약 + 실무 코드

In [ ]:
# 필요한 패키지 설치 (최초 1회만 실행)
# pip install konlpy soynlp scikit-learn numpy

## 1. 한국어 자연어 처리의 특성 (개념만, 코드 없음)

- 한국어는 **교착어**: 의미적 기능(어간)과 문법적 기능(조사/어미)이 붙어서 한 단어가 됨
  - 엘리스**는** / 엘리스**가** / 엘리스**에게** → 의미 핵심은 전부 "엘리스"
  - 먹**다** / 먹**었다** / 먹**는다** → 어간 "먹" + 어미
- 그 결과 **단어의 정의 자체가 불명확** → 띄어쓰기만으로 단어를 못 나눔
- 그래서 자연어 처리의 첫 단계는 항상 **형태소 분석** (의미 단위로 쪼개기)

> 이 부분은 형태소 분석이 왜 필요한지에 대한 이유이므로, 코드가 없어도 반드시 기억.

## 2. KoNLPy — 형태소 분석기 모음

KoNLPy는 자체 엔진이 아니라 **5개의 서로 다른 형태소 분석기를 통일된 인터페이스로 감싼 래퍼**:

| 분석기 | 특징 |
|---|---|
| Mecab | 속도 빠름 |
| 한나눔(Hannanum) | 태그 9개 |
| 꼬꼬마(Kkma) | 태그 56개, 문장 분리(`sentences`) 지원 |
| Komoran | 사용자 사전(`userdict=`) 추가 가능 |
| Open Korean Text(Okt) | 균형 잡힌 성능, 어간 복원(`stem=True`) 가능 |

**각 분석기는 사전 기반**이라 결과와 태그 체계가 서로 다름 → 어떤 분석기를 쓰느냐로 결과가 달라진다는 것 자체가 핵심.

In [ ]:
from konlpy.tag import Kkma, Okt

kkma = Kkma()
okt = Okt()

sent = "안녕 나는 엘리스야 너 이름 뭐야?"

# 명사만 추출
print("Kkma nouns:", kkma.nouns(sent))
print("Okt nouns :", okt.nouns(sent))

# 품사 태깅 (분석기마다 태그 체계 다름: Kkma는 NNG/NP/JX 식, Okt는 Noun/Josa 식)
print("Kkma pos:", kkma.pos(sent))
print("Okt pos :", okt.pos(sent))

# Okt는 stem=True로 어간 복원 가능 (예: '반가워' -> ('반갑다', 'Adjective'))
print("Okt stem:", okt.pos("만나서 반가워요!", stem=True))

# 문장 분리는 Kkma만 지원
print("Kkma sentences:", kkma.sentences("오늘 날씨 좋다. 산책 가자."))


## 3. soynlp — 미등록 단어(OOV) 문제 해결

**문제**: KoNLPy 계열은 전부 사전 기반이라, 사전에 없는 신조어/고유명사가 나오면 이상하게 쪼갬.

예시(PDF): "보코하람"(테러조직명)
- 꼬꼬마 → `[보, 보코, 코, 테러, 소말리, 전쟁]` (엉망으로 분리됨)
- Okt → `[보코하람, 테러, 소말리아, 전쟁]` (그나마 낫지만 항상 보장되진 않음)

**해결 아이디어**: 사전을 안 쓰고, 학습 코퍼스 안에서 "이 글자들이 얼마나 자주 붙어서 나오는가"라는 **통계**만으로 단어 경계를 스스로 학습. 코퍼스에 자주 등장하기만 하면 사전에 없어도 잡아낼 수 있음.

> 실무에서는 KoNLPy 단독이 아니라 **"soynlp로 신조어 뽑기 → Komoran 사용자 사전에 추가 → KoNLPy로 최종 분석"** 식으로 두 개를 같이 씀.

In [ ]:
from soynlp.utils import DoublespaceLineCorpus
from soynlp.word import WordExtractor
from soynlp.noun import LRNounExtractor_v2

# 한 줄에 문서(문장) 하나씩 들어있는 텍스트 파일 경로
corpus_path = "학습데이터_경로.txt"
train_data = DoublespaceLineCorpus(corpus_path)

# 명사 추출기: 코퍼스 통계만으로 명사 후보를 스스로 학습 (사전 불필요)
noun_extractor = LRNounExtractor_v2()
nouns = noun_extractor.train_extract(train_data)  # {단어: NounScore(빈도, 점수 등)}

# 단어 추출기: 어절을 좌(의미)-우(문법) 구조로 쪼개서 통계적 점수 계산
word_extractor = WordExtractor()
words = word_extractor.train_extract(train_data)


## 4. 문장 유사도 — 자카드 지수 & 코사인 유사도

### 자카드 지수 (Jaccard Index)
두 문장의 **공통 단어 수 / 전체 고유 단어 수**

$$J(A,B) = \frac{|A \cap B|}{|A \cup B|}$$

예시(PDF): 날씨 관련 두 문장, 공통 단어 2개 / 전체 고유 단어 8개 → **2/8 = 0.25**

### 코사인 유사도 (Cosine Similarity)
두 벡터 사이의 **각도**만 비교 (크기 무시). 1에 가까울수록 유사.

$$\cos(\theta) = \frac{A \cdot B}{\|A\|\|B\|}$$

예시(PDF): A=[1,3], B=[0,2]

$$\cos(\theta) = \frac{(1\times0)+(3\times2)}{\sqrt{1^2+3^2}\times\sqrt{0^2+2^2}} = \frac{6}{2\sqrt{10}} \approx 0.9487$$

**실무 선택 기준**: 자카드는 계산이 단순하지만 문장이 길어지면 왜곡되기 쉬움 → 간단한 프로토타입용. 코사인은 임베딩/TF-IDF 벡터와 함께 쓰기 좋아서 검색·추천·챗봇 매칭 등 **실무 표준**은 코사인 쪽.

In [ ]:
def jaccard_similarity(sent1, sent2):
  # 단순 공백 기준 토큰화 (실무에서는 형태소 분석기로 토큰화하는 게 더 정확함)
  set1 = set(sent1.split())
  set2 = set(sent2.split())
  intersection = set1 & set2
  union = set1 | set2
  return len(intersection) / len(union)

s1 = "오늘 날씨가 정말 좋네요 산책 가고 싶어요"
s2 = "오늘 날씨는 흐리고 비가 올 것 같아요"
print("자카드 유사도:", jaccard_similarity(s1, s2))


In [ ]:
import numpy as np

def cosine_similarity(vec1, vec2):
  vec1 = np.array(vec1)
  vec2 = np.array(vec2)
  dot_product = np.dot(vec1, vec2)
  norm1 = np.linalg.norm(vec1)  # 벡터 크기(magnitude)
  norm2 = np.linalg.norm(vec2)
  return dot_product / (norm1 * norm2)

# PDF 손계산 예시 검증: 6 / (2*sqrt(10)) ~= 0.9487
print("코사인 유사도 (예시):", cosine_similarity([1, 3], [0, 2]))


In [ ]:
# 실무: 문장을 벡터화한 뒤 sklearn 내장 함수로 바로 코사인 유사도 계산
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine_similarity

cv = CountVectorizer()
vectors = cv.fit_transform([s1, s2]).toarray()

print("문장 벡터 코사인 유사도:", sk_cosine_similarity([vectors[0]], [vectors[1]]))


## 참고: 코드로 안 옮겨진 개념 (짧게 짚고 넘어가기)

- **유클리드 거리**: $d(p,q) = \sqrt{\sum (q_i - p_i)^2}$ — 벡터 간 직선 거리. PDF에도 코드 없이 언급만 있었고, 문장 유사도보다는 다른 거리 계산에 더 쓰임. 한 줄 요약으로 충분.
- **CNN/RNN 기반 문장 임베딩**: PDF에 다이어그램만 있고 코드 없음 → 딥러닝 파트라 실습 시 별도로 다룰 부분, 여기서는 생략.

In [ ]:
# ===== 1. Bag of words (BoW) - 단어 빈도수 기반 문서 벡터화 =====

from sklearn.feature_extraction.text import CountVectorizer  # BoW 벡터를 만들어주는 sklearn 클래스를 불러옴

docs = [
  "오늘 날씨가 정말 좋네요 산책 가고 싶어요",  # 문서 1
  "오늘 날씨는 흐리고 비가 올 것 같아요"      # 문서 2
]

cv = CountVectorizer()  # BoW 벡터화기 객체 생성 (기본은 unigram, 즉 단어 1개 단위)
bow_matrix = cv.fit_transform(docs)  # 문서 집합 전체에서 단어 사전을 학습하고, 각 문서를 벡터로 변환

print(cv.get_feature_names_out())  # 벡터의 각 차원이 어떤 단어에 대응하는지 확인 (= 전체 단어 목록)
print(bow_matrix.toarray())  # 문서별 단어 빈도수 벡터 확인 (희소 행렬을 일반 배열로 변환해서 출력)
# -> 벡터의 차원 수 = 전체 데이터에 등장한 단어의 개수 (PDF에서 본 "차원 = 모든 단어의 개수"가 바로 이 부분)
# -> 대부분의 값이 0으로 채워짐 (PDF에서 본 "희소(sparse) 벡터" 문제)

In [ ]:
# ===== 2. Bag of N-grams - 연속된 N개 단어를 기준으로 벡터화 =====

bigram_cv = CountVectorizer(ngram_range=(2, 2))  # ngram_range=(2,2)로 설정하면 bi-gram(연속 단어 2개)만 사용
bigram_matrix = bigram_cv.fit_transform(docs)  # bi-gram 기준으로 문서 벡터 생성

print(bigram_cv.get_feature_names_out())  # "오늘 날씨가", "날씨가 정말" 같은 2단어 묶음이 하나의 단위가 됨
print(bigram_matrix.toarray())  # bi-gram 기준 문서 벡터 확인

# unigram + bigram을 동시에 쓰고 싶으면 ngram_range=(1, 2)로 설정
uni_bi_cv = CountVectorizer(ngram_range=(1, 2))  # 단어 1개짜리와 2개짜리를 모두 벡터에 포함
uni_bi_matrix = uni_bi_cv.fit_transform(docs)  # unigram + bigram 혼합 벡터 생성
print(uni_bi_cv.get_feature_names_out())  # 단어 1개 묶음과 2개 묶음이 함께 나옴

In [ ]:
# ===== 3. TF-IDF - 자주 나오지만 여러 문서에 흔한 단어의 가중치는 낮추는 방식 =====

from sklearn.feature_extraction.text import TfidfVectorizer  # TF-IDF 벡터화 클래스를 불러옴

tfidf = TfidfVectorizer()  # TF-IDF 벡터화기 객체 생성
tfidf_matrix = tfidf.fit_transform(docs)  # 문서 집합에서 TF-IDF 점수를 계산해 벡터로 변환

print(tfidf.get_feature_names_out())  # 각 차원에 대응하는 단어 목록 확인
print(tfidf_matrix.toarray())  # 문서별 TF-IDF 점수 벡터 확인
# -> TF(단어 빈도) x IDF(그 단어가 등장하는 문서가 적을수록 커지는 값)
# -> "오늘"처럼 두 문서에 다 나오는 단어는 점수가 낮아지고, 한 문서에만 나오는 단어는 점수가 높아짐
# -> PDF에서 본 "자주 발생하는 단어가 항상 문서의 특징을 잘 나타내는 건 아니다"는 문제를 이 가중치로 보완

In [ ]:
# ===== 4. doc2vec - 신경망 기반으로 문서를 저차원 벡터로 임베딩 =====

from gensim.models.doc2vec import Doc2Vec, TaggedDocument  # doc2vec 모델과, 문서에 태그(id)를 붙이는 클래스를 불러옴

raw_docs = [
  "오늘 날씨가 정말 좋네요 산책 가고 싶어요",  # 문서 1 원문
  "오늘 날씨는 흐리고 비가 올 것 같아요"      # 문서 2 원문
]

tagged_data = [
  TaggedDocument(words=doc.split(), tags=[str(i)])  # 각 문서를 단어 리스트로 쪼개고, 문서 번호를 태그로 붙임
  for i, doc in enumerate(raw_docs)  # 문서 인덱스(0, 1, ...)를 태그 값으로 사용
]

model = Doc2Vec(
  tagged_data,      # 학습에 사용할 태그 달린 문서 목록
  vector_size=20,   # 문서 벡터의 차원 수 (BoW처럼 전체 단어 수만큼이 아니라, 원하는 만큼 저차원으로 설정 가능)
  window=2,         # 문맥으로 볼 앞뒤 단어의 범위
  min_count=1,      # 최소 이 횟수 이상 등장한 단어만 학습에 사용 (데이터가 적어서 1로 설정)
  epochs=40         # 전체 데이터를 반복 학습할 횟수
)

doc1_vector = model.dv["0"]  # 학습된 문서 1(태그 "0")의 임베딩 벡터를 가져옴
print(doc1_vector.shape)  # (20,) 출력 -> BoW와 달리 전체 단어 수와 무관하게 20차원으로 고정됨
print(doc1_vector)  # 저차원 실수 벡터 확인 (BoW처럼 대부분 0인 희소 벡터가 아니라 촘촘한 벡터)

similar_docs = model.dv.most_similar("0")  # 문서 1과 벡터 공간에서 가장 가까운(유사한) 문서를 찾음
print(similar_docs)  # (문서 태그, 유사도) 쌍의 리스트로 출력